In [ ]:
# import libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import math

evaluation_file = "run63_medium_10fact_mega_shared"
model_name = "run53_mix_mega_shared_simple_model"

normalize= 0
os.environ["CUDA_VISIBLE_DEVICES"]="-1"
number_of_detectors = 6
data_dir = "/data/test_newrepo"


In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
file_path = data_dir+'/'+evaluation_file+'_dataset.pkl'

with open(file_path, 'rb') as file:
    evaluation_array = pickle.load(file)
    

In [ ]:
evaluation_array.shape

In [ ]:
def diff_phi(a1, a2):
    diff = abs(a1 - a2)
    if diff > 180:
        diff = 360 - diff
    return diff

def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg


def l2_normalize(data):
  """
  Normalize a NumPy array using the L2 (Euclidean) norm.

  Args:
    data (numpy.ndarray): The array to normalize. Can be a 1D vector
                         or a 2D matrix (where each row is a vector to normalize).

  Returns:
    numpy.ndarray: The L2-normalized array.
  """

  data = np.array(data)
  if data.ndim == 1:
    # Case: 1D vector
    norm = np.sqrt(np.sum(data**2))
    if norm == 0:
      return data  # Avoid division by zero if the vector is null
    return data / norm
  elif data.ndim == 2:
    # Case: 2D matrix (normalize each row)
    norms = np.sqrt(np.sum(data**2, axis=1, keepdims=True))
    # Handle the case of null norms (zero rows)
    norms[norms == 0] = 1
    return data / norms
  else:
    raise ValueError("Input must be a 1D or 2D array.")

    
def prepare_dataset(data_array):
  """
  Prepare the dataset and corresponding labels for training or evaluation.

  Steps performed:
    1. Normalize the detector counts using L2 normalization.
    2. Extract longitude and latitude labels.
    3. Optionally scale labels between 0 and 1 (if `normalize == 1`).
    4. Convert labels to radians for further processing.

  Args:
    data_array (list of dict): Each element must contain:
        - 'counts' (numpy.ndarray): Detector count values.
        - 'coord' (tuple or list): Coordinates (longitude, latitude).

  Returns:
    tuple:
      - dataset (numpy.ndarray): Normalized detector counts.
      - labels (numpy.ndarray): Original labels (longitude, latitude).
      - labels_norm (numpy.ndarray): Scaled labels (if normalization applied).
      - theta (tuple): Extracted theta values in degrees.
      - phi (tuple): Extracted phi values in degrees.
      - coords_rad (list): List of coordinates in radians.
  """

  dataset = np.empty((len(data_array), number_of_detectors))
  labels = np.empty((len(data_array), 2))
    
  count = -1
  for element in data_array:
      count += 1
      dataset[count] = l2_normalize(element['counts'])
      labels[count][0] = float(element['coord'][0])
      labels[count][1] = float(element['coord'][1])
        
  labels = labels[:count+1]
  dataset = dataset[:count+1]

    
  labels_norm = np.empty((len(dataset), 2))
    
  count = -1
  for element in labels:
      count += 1
      labels_norm[count][0] = labels[count][0]
      labels_norm[count][1] = labels[count][1]

  # Convert to radians
  coords_rad = []
    
  for theta, phi in labels:
      coords_rad.append([theta, phi])

  theta, phi = zip(*coords_rad)
    
  return dataset, labels, labels_norm, theta, phi, coords_rad



In [ ]:
evaluation_array_flatten = flattened = evaluation_array.flatten()
print(evaluation_array_flatten.shape)
test_dataset, test_labels_raw, test_labels_norm, test_theta, test_phi, test_coords_rad = prepare_dataset(evaluation_array_flatten)
print(test_labels_norm.shape)
test_labels = test_labels_norm

In [ ]:

plt.hist(test_theta,alpha=0.5)
plt.xlabel("Theta")
plt.ylabel("Counts")

In [ ]:

plt.hist(test_phi,alpha=0.5)
plt.xlabel("Phi")
plt.ylabel("Counts")

In [ ]:

plt.hist(test_labels_norm[:,0],alpha=0.5)


In [ ]:

plt.hist(test_labels_norm[:,1],alpha=0.5)


In [ ]:
from tensorflow.keras.models import load_model

# Load the model back from the saved directory
model = load_model(data_dir+"/"+model_name+".keras")

In [ ]:
# Process testing dataset
pred_data = model.predict(test_dataset)

In [ ]:
pred_data.shape

In [ ]:
max_lon = 180.0
min_lon = 0.0
    
max_lat = 360.0
min_lat = 0.0

pred_data_original = np.empty((len(pred_data),2))

if normalize==1:
    pred_data_original[:, 0] = pred_data[:, 0]*(max_lon-min_lon)+min_lon
    pred_data_original[:, 1] = pred_data[:, 1]*(max_lat-min_lat)+min_lat
else:
    pred_data_original[:, 0] = pred_data[:, 0]
    pred_data_original[:, 1] = pred_data[:, 1]

test_labels_original = np.empty((len(test_labels),2))

if normalize==1:
    test_labels_original[:, 0] = test_labels[:, 0]*(max_lon-min_lon)+min_lon
    test_labels_original[:, 1] = test_labels[:, 1]*(max_lat-min_lat)+min_lat
else:
    test_labels_original[:, 0] = test_labels[:, 0]
    test_labels_original[:, 1] = test_labels[:, 1]


absolute_diffs_theta = np.abs(pred_data_original[:, 0] - test_labels_original[:, 0])
absolute_diffs_phi = np.abs(pred_data_original[:, 1] - test_labels_original[:, 1])

# Calculate the Mean Absolute Error (MAE) for each element separately
mae_theta = np.mean(absolute_diffs_theta)
mae_phi = np.mean(absolute_diffs_phi)

print("Mean Absolute Error for Theta:", mae_theta)
print("Mean Absolute Error for Phi:", mae_phi)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 0],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data_original[:,0],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta")
plt.legend()
plt.ylabel("Counts")

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 1],bins=50,alpha=0.6,color="b",label="reco coords")
plt.hist(pred_data_original[:,1],bins=50,alpha=0.6,color="r",label="test coords")
plt.xlabel("Phi")
plt.legend()
plt.ylabel("Counts")

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,0],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 0],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta sin")
plt.legend()
plt.ylabel("Counts")

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,1],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 1],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta cos")
plt.legend()
plt.ylabel("Counts")

In [ ]:
pred_data_original_reshaped = pred_data_original.reshape(evaluation_array.shape[0], evaluation_array.shape[1], 2)
test_labels_original_reshaped = test_labels_original.reshape(evaluation_array.shape[0], evaluation_array.shape[1], 2)

In [ ]:
# Calculate the mean distances between corresponding coordinates considering that we have multiple GRBs in the same position.

distances = []
theta_distances = []
phi_distances = []

for i in range(0,len(pred_data_original_reshaped)):
    
    distances_partial = []
    theta_distances_partial = []
    phi_distances_partial = []
    
    for j in range(0,len(pred_data_original_reshaped[i])):
    
        d = angular_distance(pred_data_original_reshaped[i][j][0],pred_data_original_reshaped[i][j][1],test_labels_original_reshaped[i][j][0],test_labels_original_reshaped[i][j][1])
        theta_dist = np.abs(pred_data_original_reshaped[i][j][0]-test_labels_original_reshaped[i][j][0])
        phi_dist = diff_phi(pred_data_original_reshaped[i][j][1],test_labels_original_reshaped[i][j][1])
        distances_partial.append(d)
        theta_distances_partial.append(theta_dist)
        phi_distances_partial.append(phi_dist)
    
    distances_avg = np.mean(distances_partial)
    theta_distances_avg = np.mean(theta_distances_partial)
    phi_distances_avg = np.mean(phi_distances_partial)
 
    distances.append(distances_avg)
    theta_distances.append(theta_distances_avg)
    phi_distances.append(phi_distances_avg)

In [ ]:
import pickle
if False:
    # Store the data for further plotting
    with open(data_dir+"/"+evaluation_file+"_distances_model_nobkg_plot.pkl", "wb") as f:
        pickle.dump(distances, f)

In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

nside = 16
npix = hp.nside2npix(nside)
m = np.array(distances)
new_m = np.sqrt(m)
hp.projview(
    m,
   
    coord=["G"],
    graticule=True,
    graticule_labels=True,
    unit="cbar label",
    xlabel="longitude",
    ylabel="latitude",
    cb_orientation="vertical",
    latitude_grid_spacing=30,
    projection_type="aitoff",
    title="Aitoff projection",
    cmap="turbo",
    nest=True
)

plt.show()


In [ ]:
print(np.mean(distances))
print(np.mean(theta_distances))
print(np.mean(phi_distances))